In [1]:
from langgraph.graph import StateGraph,START,END,add_messages

from pydantic import BaseModel
from typing import Annotated
from dotenv import load_dotenv
from src.apps.agent.llm import get_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode,tools_condition
load_dotenv()


True

In [2]:
from langgraph.graph import StateGraph,START,END,add_messages
from langchain_core.messages import AnyMessage,AIMessage, HumanMessage
from typing_extensions import TypedDict
from typing import Annotated
from dotenv import load_dotenv
from src.apps.agent.llm import get_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode,tools_condition
from langgraph.types import interrupt,Command
checkpointer=InMemorySaver()


llm = get_model()

load_dotenv()


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]



def send_email(state:State, email: str) -> str:
    """send email"""
    dicision = interrupt(
        {
            "type":"approval",
            "reason":"For sending email",
            "question":state['messages'][-1].content,
            "instruction":"Approve this action? yes/no"
        }
    )
    if dicision['approved'] == 'no':
        return {"message"[AIMessage(content="Not approved")]}
    
    return f"email send {email}"


model = llm.bind_tools([send_email])



def chat(state: State):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(State)
graph.add_node('chat',chat)
graph.add_node('tool',ToolNode([send_email]))
graph.add_edge(START,'chat')
graph.add_conditional_edges(
    'chat',tools_condition,{
        "tools":"tool",
        '__end__':END
    }
)

graph.add_edge('tool','chat')



agent = graph.compile(checkpointer=checkpointer)



In [ ]:



response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to anshu@gmail.com saying hii"
            )
        ]
    },
    config={"configurable": {"thread_id": "12"}}
)

print(response)
if response.get("__interrupt__"):
    print(response["__interrupt__"][0].value)
    
else:
    print(response["messages"][-1].content)

{'messages': [HumanMessage(content='Send an email to anshu@gmail.com saying hii', additional_kwargs={}, response_metadata={}, id='fcf02c80-5c32-4023-b8e6-97e8c5516e28'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to anshu@gmail.com with content "hii". Use the send_email tool. Need to provide email and state with messages array. Probably just a simple message. Provide messages: maybe a HumanMessage? But we can just include a simple message content. The schema expects state.messages: array of messages (AIMessage, HumanMessage, etc). We can just provide a simple object with type "human" and content "hii". Or maybe just a string? The type expects Message objects. Let\'s construct a HumanMessage: {type: "human", content: "hii"}.\n\nThus call function.', 'tool_calls': [{'id': 'fc_e79ac63c-848d-44bf-9aa8-44e0201044da', 'function': {'arguments': '{"email":"anshu@gmail.com","state":{"messages":[{"content":"hii","type":"human"}]}}', 'name': 'se

In [6]:
agent.invoke(Command(resume={"approved":input()}),config={"configurable": {"thread_id": "12"}})

{'messages': [HumanMessage(content='Send an email to anshu@gmail.com saying hii', additional_kwargs={}, response_metadata={}, id='fcf02c80-5c32-4023-b8e6-97e8c5516e28'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to anshu@gmail.com with content "hii". Use the send_email tool. Need to provide email and state with messages array. Probably just a simple message. Provide messages: maybe a HumanMessage? But we can just include a simple message content. The schema expects state.messages: array of messages (AIMessage, HumanMessage, etc). We can just provide a simple object with type "human" and content "hii". Or maybe just a string? The type expects Message objects. Let\'s construct a HumanMessage: {type: "human", content: "hii"}.\n\nThus call function.', 'tool_calls': [{'id': 'fc_e79ac63c-848d-44bf-9aa8-44e0201044da', 'function': {'arguments': '{"email":"anshu@gmail.com","state":{"messages":[{"content":"hii","type":"human"}]}}', 'name': '